# News Summarization

### Scrape the article you want to summarize

In [4]:
!pip install lxml_html_clean  
#Already installed

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 3.4 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/4.0 MB 3.7 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.0 MB 4.1 MB/s eta 0:00:01
   ------------------------------- -------- 3.1/4.0 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 4.1 MB/s  0:00:01

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 2/2 [lxml_html_clean]



In [9]:
%pip install newspaper3k

  Using cached newspaper3k-0.2.8-py3-none-any.whl.metadata (11 kB)
  Using cached cssselect-1.4.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached tldextract-5.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached feedfinder2-0.0.4-py3-none-any.whl
  Using cached jieba3k-0.35.1-py3-none-any.whl
  Using cached tinysegmenter-0.3-py3-none-any.whl
  Using cached requests_file-3.0.1-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached newspaper3k-0.2.8-py3-none-any.whl (211 kB)
Using cached cssselect-1.4.0-py3-none-any.whl (18 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached tldextract-5.3.1-py3-none-any.whl (105 kB)
Using cached requests_file-3.0.1-py2.py3-none-any.whl (4.5 kB)

   ----- ---------------------------------- 1/8 [jieba3k]
   ----- ---------------------------------- 1/8 [jieba3k]
   ----- ---------------------------------- 1/8 [jieba3k]
   ----- ---------------------------------- 1/8 [jieba3k]
   --

In [10]:
from scraping import return_single_article
article = return_single_article('https://www.cnn.com/2020/10/20/politics/joe-biden-tax-plan/index.html')

In [11]:
print(article['title'], '\n\n')
print(article['authors'], '\n\n')
print(article['source'], '\n\n')
# Printing just the first 1000 characters so it doesn't flood your notebook screen
print("ARTICLE PREVIEW:\n", article['article'][:1000], '...\n')
# print(article['article'], '\n\n')

Joe Biden tax plan: What you need to know 


By Katie Lobosco 


CNN 


ARTICLE PREVIEW:
 Washington —

Democratic presidential candidate Joe Biden has put forth several proposals that would change the tax code.

In general, he’s proposing to raise taxes on the wealthy and on corporations by reversing some of the Republican-backed tax cuts that President Donald Trump signed into law in 2017.

It’s unlikely that Biden’s campaign plans would come to fruition just as he’s proposed them, even if he wins the election. He’d have an easier time getting them passed if Democrats also take back the Senate and maintain control of the House.

Here’s what you need to know:

Biden pledges not raise taxes on anyone earning less than $400,000

Biden has pledged not to raise taxes on those earning less than $400,000 a year (that’s more than 90% of taxpayers). When considering direct taxes only, several economic models show that to be true, including one from the bipartisan Committee for a Responsible F

### Summarization

In [12]:
from hf_summarizer import bart_summarize

In [13]:
# this function uses this model: https://huggingface.co/sshleifer/distilbart-cnn-12-6
# which is a bert-like model trained on the cnn-dailymail dataset
# its one of the best summarization models available in transformers, very fast, good for deployment
summary = bart_summarize(article['article'])

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [14]:
summary

' Democratic presidential candidate Joe Biden has put forth several proposals that would change the tax code . He’s proposing to raise taxes on the wealthy and on corporations by reversing some of the Republican-backed tax cuts that President Donald Trump signed into law in 2017 . Biden has pledged not to increase taxes on those earning less than $400,000 a year .'

#### There are many other summary options available in hf_summarizer.py including different pegasus models, t5 and methods that address the issues with these summarizers (like the fact that some of these summarizers can only have limited size inputs)

### Statistical Summarization

In [19]:
from statistical_summarize import run_statistical_summarizers, run_tf_idf_summarization

In [16]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\piyus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [20]:
tfidf_summary = run_tf_idf_summarization(text=article['article'], num_sentences=5)
# If tfidf_summary returns a list of sentences, join them into a paragraph for evaluation
if type(tfidf_summary) is list:
    tfidf_summary = ' '.join(tfidf_summary)

print("--- TF-IDF SUMMARY ---")
print(tfidf_summary, '\n')
# run_statistical_summarizers(text=article['article'], num_sentences=5)

--- TF-IDF SUMMARY ---
That means those taxpayers won’t be writing a bigger check to the IRS. Economists assume that workers eventually bear some of the cost of those taxes. This would affect those with taxable incomes above $400,000. The right-leaning American Enterprise Institute found they would raise $2.8 trillion over a decade. The Tax Policy Center put the number at $2.4 trillion.  



## Evaluation

In [27]:
%pip install rouge-score

  Using cached rouge_score-0.1.2-py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


In [29]:
from rouge_score import rouge_scorer
from evaluation import display_rouge_scores

In [ ]:
# # Create a temporary Article object to trigger newspaper's built-in NLP
# from newspaper import Article as NewspapreArticle

# temp_article = NewspapreArticle('https://www.cnn.com/2020/10/20/politics/joe-biden-tax-plan/index.html') # or whatever URL the professor gives
# temp_article.download()
# temp_article.parse()
# temp_article.nlp()

# # This becomes your auto-generated baseline/reference summary!
# reference_summary = temp_article.summary
# print("AUTOMATED REFERENCE SUMMARY:\n", reference_summary)

In [30]:
# Ground truth summary for the CNN Biden Tax Plan article
reference_summary = "Joe Biden's proposed tax plan aims to raise taxes on corporations and wealthy Americans earning over $400,000 a year, while avoiding tax increases for middle-class earners. The plan seeks to fund new investments in infrastructure, education, and healthcare."

# Evaluate both models side-by-side
display_rouge_scores(reference_summary, summary, model_name="Hybrid BART")
display_rouge_scores(reference_summary, tfidf_summary, model_name="Statistical (TF-IDF)")

--- ROUGE Evaluation for Hybrid BART ---
        Precision  Recall  F1-Score
rouge1     0.3548  0.5366    0.4272
rouge2     0.1475  0.2250    0.1782
rougeL     0.2258  0.3415    0.2718
----------------------------------------
--- ROUGE Evaluation for Statistical (TF-IDF) ---
        Precision  Recall  F1-Score
rouge1     0.1562  0.2439    0.1905
rouge2     0.0159  0.0250    0.0194
rougeL     0.1094  0.1707    0.1333
----------------------------------------


,Precision,Recall,F1-Score
rouge1,0.1562,0.2439,0.1905
rouge2,0.0159,0.0250,0.0194
rougeL,0.1094,0.1707,0.1333


#### These statistical techniques do not use any Machine Learning algorithms but are very fast and give a decent result

### Sentiment Analysis

In [15]:
from sentiment_analysis import hf_topn_sentiment

In [16]:
# this function returns the top positive and top negative sentences from a piece of text,
# not exactly summarization but can get the most polarizing lines from an article
top_positive, top_negative = hf_topn_sentiment(article['article'])

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


...

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [17]:
top_positive

[(0.9544905424118042,
  'Biden is also proposing to expand the child tax credit and to reestablish a first-time homebuyers’ tax credit.'),
 (0.9975979924201965,
  'When considering direct taxes only, several economic models show that to be true, including one from the bipartisan Committee for a Responsible Federal Budget and the Penn Wharton Budget Model.')]

In [18]:
top_negative

[(0.9991311430931091,
  'He’d have an easier time getting them passed if Democrats also take back the Senate and maintain control of the House.'),
 (0.9991363883018494,
  'Biden’s campaign proposal is vague on some key details, but here’s how it could work: The current system – which allows savers to take up to $19,500 in income-tax deductions every year – would be replaced with a flat refundable tax credit.')]